In [ ]:
# Import utility functions
import sys
import pandas as pd
import geopandas as gpd
import numpy as np
import os
import requests
import json
sys.path.insert(0, '../..')  # Add parent directory to path

# Import from local utility modules
from lvt_utils import (model_split_rate_tax, calculate_current_tax, ensure_geodataframe, 
                       categorize_property_type, calculate_category_tax_summary, print_category_tax_summary)
from census_utils import (get_census_data_with_boundaries, match_parcels_to_demographics, 
                          create_demographic_summary, calculate_median_percentage_by_quintile, match_to_census_blockgroups)
from viz import (create_scatter_plot, plot_comparison, calculate_correlations, weighted_median, 
                 create_quintile_summary, plot_quintile_analysis, create_property_category_chart, 
                 create_map_visualization, calculate_block_group_summary, filter_data_for_analysis)

# Control variable for data scraping
data_scrape = 0  # Set to 1 to scrape new data, 0 to use existing data

print("✅ Utility functions imported from LVTShift modules")


In [ ]:
import glob
from datetime import datetime

# Data directories
data_dir = 'data'
os.makedirs(data_dir, exist_ok=True)

# --- URLs for Rochester Open Data (ArcGIS REST) ---
PARCELS_URL = "https://maps.cityofrochester.gov/arcgis/rest/services/App_PropertyInformation/ROC_Parcel_Query_RPS_Merged/MapServer/0/query"
EXEMPTIONS_URL = "https://maps.cityofrochester.gov/arcgis/rest/services/App_PropertyInformation/ROC_Parcel_Query_RPS_Merged/MapServer/8/query"
TAXBILL_URL = "https://maps.cityofrochester.gov/arcgis/rest/services/App_PropertyInformation/ROC_Parcel_Query_RPS_Merged/MapServer/9/query"

def download_arcgis_layer(url, id_col=None, geo=True, max_records=10000):
    """
    Download an entire ArcGIS REST layer (table or geojson) in batches.
    """
    import pandas as pd
    import geopandas as gpd

    # First grab record count
    r = requests.get(url, params={"where":"1=1","returnCountOnly":"true","f":"json"})
    n_records = r.json()["count"]

    dfs = []
    for start in range(0, n_records, max_records):
        params = {
            "where": "1=1",
            "outFields": "*",
            "f": "geojson" if geo else "json",
            "resultOffset": start,
            "resultRecordCount": min(max_records, n_records - start),
            "returnGeometry": str(geo).lower()
        }
        resp = requests.get(url, params=params)
        if geo:
            gdf = gpd.GeoDataFrame.from_features(resp.json()["features"])
        else:
            records = resp.json()["features"] if "features" in resp.json() else []
            # ArcGIS tables return JSON with features > attributes
            attr = [rec["attributes"] for rec in records]
            gdf = pd.DataFrame(attr)
        dfs.append(gdf)
        print(f"Downloaded {(start+1)}–{start+params['resultRecordCount']} of {n_records}", end="\r")
    df = pd.concat(dfs, ignore_index=True)
    return df

if data_scrape == 1:
    print("🔄 Downloading fresh Rochester property data...")

    # Download parcels (GeoDataFrame)
    print("📥 Downloading parcels...")
    gdf = download_arcgis_layer(PARCELS_URL, geo=True)
    print(f"✅ Downloaded {len(gdf):,} Rochester parcels")

    # Download exemptions (DataFrame, join by SBL20 to PARCELID)
    print("📥 Downloading exemptions...")
    exemptions = download_arcgis_layer(EXEMPTIONS_URL, geo=False)
    # Clean and sum multiple records for the same SBL (if present)
    exemptions["ExemptionAmount"] = pd.to_numeric(exemptions["ExemptionAmount"], errors="coerce").fillna(0)
    exemptions_sum = exemptions.groupby("SBL20")["ExemptionAmount"].sum().reset_index()
    exemptions_sum.rename(columns={"ExemptionAmount":"total_exemptions"}, inplace=True)

    # Download tax bill info (DataFrame, join by SBL20 to PARCELID)
    print("📥 Downloading tax bill data...")
    taxbill = download_arcgis_layer(TAXBILL_URL, geo=False)
    # Take the most recent tax bill per parcel if multiple
    taxbill_latest = taxbill.sort_values(["SBL20", "YR_2023"], ascending=[True,False])
    taxbill_latest = taxbill_latest.groupby("SBL20").first().reset_index() # Only most recent bill per parcel

    # Join the auxiliary data to main parcels
    gdf = gdf.merge(exemptions_sum, left_on="PARCELID", right_on="SBL20", how="left")
    gdf = gdf.merge(taxbill_latest, left_on="PARCELID", right_on="SBL20", how="left", suffixes=("", "_bill"))
    gdf["total_exemptions"] = gdf["total_exemptions"].fillna(0)

    # Calculate land/improvement and property tax variables based on Rochester column names
    gdf["land_value"] = gdf["CURRENT_LAND_VALUE"]
    gdf["total_value"] = gdf["CURRENT_TOTAL_VALUE"]
    gdf["taxable_value"] = gdf["CURRENT_TAXABLE_VALUE"]

    # If available, use TrueTax from bill. Otherwise, estimate using mill rate
    # Using city's stated mill rate in 2023: 20.992 per $1000 (source: https://www.cityofrochester.gov/article.aspx?id=8589935801)
    # But we use observed tax first
    gdf["property_tax"] = gdf["TrueTax"]
    missing_tax = gdf["property_tax"].isna()
    gdf.loc[missing_tax, "property_tax"] = gdf.loc[missing_tax, "taxable_value"] * (19.74/1000)

    # Compute improvement value, basic split-rate logic (if you wish)
    gdf["improvement_value"] = gdf["total_value"] - gdf["land_value"]

    # Save parquet
    today_str = datetime.now().strftime("%Y%m%d")
    save_path = f"{data_dir}rochester_parcels_processed_{today_str}.parquet"
    gdf.to_parquet(save_path)
    print(f"💾 Saved processed data to {save_path}")

else:
    print("📂 Loading existing Rochester property data...")
    # Find all processed parquet files in the data_dir
    parquet_files = glob.glob(os.path.join(data_dir, "rochester_parcels_processed_*.parquet"))
    if not parquet_files:
        print("❌ No processed Rochester data files found in data/rochester/. Please set data_scrape = 1 to download fresh data.")
        raise FileNotFoundError("No processed Rochester data files found.")
    # Extract dates and find the most recent file
    def extract_date(f):
        try:
            return datetime.strptime(os.path.basename(f).split("_")[-1].replace(".parquet", ""), "%Y%m%d")
        except Exception:
            return datetime.min
    parquet_files_sorted = sorted(parquet_files, key=extract_date, reverse=True)
    most_recent_file = parquet_files_sorted[0]
    gdf = gpd.read_parquet(most_recent_file)
    print(f"✅ Loaded processed Rochester data from {most_recent_file}")

print(f"\n📊 Dataset Overview:")
print(f"Total parcels: {len(gdf):,}")
print(f"Columns: {len(gdf.columns)}")
if isinstance(gdf, gpd.GeoDataFrame):
    print(f"Geometry type: {gdf.geometry.geom_type.iloc[0]}")

# Display key statistics
total_current_revenue = gdf['property_tax'].sum()
print(f"\n💰 Current Tax System:")
print(f"Total annual revenue: ${total_current_revenue:,.2f}")
print(f"Mean property tax: ${gdf['property_tax'].mean():,.2f}")
print(f"Median property tax: ${gdf['property_tax'].median():.2f}")


In [ ]:
# Print out the median property tax rate by the first number of CLASSCD

# Ensure CLASSCD exists and is string
if "CLASSCD" in gdf.columns:
    classcd_first_digit = gdf["CLASSCD"].astype(str).str[0]
    gdf["CLASSCD_FIRST"] = classcd_first_digit

    # Compute tax_rate per parcel if not already done
    if "tax_rate" not in gdf.columns:
        gdf['tax_rate'] = gdf['CurrentTax'] / gdf['taxable_value']

    # Drop inf/-inf and nan for a clean median
    gdf_clean = gdf.replace([float('inf'), float('-inf')], pd.NA).dropna(subset=["tax_rate", "CLASSCD_FIRST"])
    medians_by_classcd = gdf_clean.groupby("CLASSCD_FIRST")["tax_rate"].median().sort_index()
    print("Median property tax rate (property_tax / taxable_value) by first digit of CLASSCD:")
    print(medians_by_classcd)
else:
    print("CLASSCD column is missing from the dataset.")


In [ ]:
# Create a homestead flag (1 if CLASSCD is in homestead_codes, else 0)
homestead_codes = [
    '210', '215', '220', '230', '240', '241', '260', '270',
    '311', '312', '314', '322'
    # 412 (condos) omitted per local law/practice as described in prompt
]

gdf['homestead_flag'] = gdf['CLASSCD'].astype(str).isin(homestead_codes).astype(int)
print(f"Homestead flag created. Number of homestead parcels: {gdf['homestead_flag'].sum():,}")


In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)
from IPython.display import display

display(gdf.head(5))


In [ ]:
# Describe CLASSDSCRP with counts, showing all rows
print("Distribution of CLASSDSCRP (property class descriptions):")
pd.set_option('display.max_rows', None)
classdscrp_counts = gdf['CLASSDSCRP'].value_counts(dropna=False)
display(classdscrp_counts)
pd.reset_option('display.max_rows')


In [ ]:
# Map CLASSDSCRP to broader property categories specific to Rochester.
# Major residential groups: Single Family, 2-4 Unit, Rowhomes, Large Apartments, Other Residential
# Also define: Mixed-Use, Vacant Land, Parking, Commercial, Industrial, Institutional, Religious, Rec, Transportation, and minimize "Other"

def aggregate_classdscrp(classdscrp):
    desc = str(classdscrp).strip().lower()
    # --- Residential ---
    if "1 family" in desc or "single family" in desc:
        return "Single Family Residential"
    if (
        ("2 family" in desc or "3 family" in desc)
        or desc in [
            "residential with improvements",
            "2 family residential", "3 family residential",
        ]
    ):
        return "2-4 Unit Residential"
    if (
        "row building" in desc
        or "rowhome" in desc
        or "detached row" in desc
        or "attached row" in desc
    ):
        return "Rowhome"
    if (
        "apartment" in desc
        or "multiple residential" in desc
        or (
            # catch "converted residential" (but don't call it just apartment, so it's under "Other Res")
            "converted residential" in desc
        )
    ):
        # Handle "Larger apartments" = traditional 'apartment' and "multiple residential", but not 2-4
        return "Large Apartment Residential" if "apartment" in desc or "multiple residential" in desc else "Other Residential"
    # Residential, Vacant Land — separate
    if "vacant" in desc or "vacant land" in desc:
        return "Residential Vacant Land" if "residential" in desc else "Vacant/Unused Land"
    # --- Mixed Use ---
    if "multi-use" in desc or "mixed use" in desc:
        return "Mixed-Use"
    # --- Parking ---
    if "parking" in desc:
        return "Parking"
    # --- Commercial/Retail/Office ---
    if ("commercial" in desc or "retail" in desc or "supermarket" in desc or "mini-mart" in desc or
        "shopping center" in desc or "restaurant" in desc or "bar" in desc or "fast food" in desc or
        "office" in desc or "bank" in desc or "dealer" in desc or "warehouse" in desc or "service" in desc or
        "hotel" in desc or "motel" in desc or "lodge" in desc or "store" in desc or "movie theater" in desc or
        "night club" in desc or "theater" in desc or "branch bank" in desc or "auto body" in desc or
        "auto dealer" in desc or "auto carwash" in desc or "mini warehouse" in desc or "fuel store" in desc or
        "gas station" in desc or "snack bar" in desc or "mall" in desc or "large retail" in desc or "billboard" in desc):
        return "Commercial / Retail / Office"
    # --- Industrial/Manufacturing/Storage/Utilities ---
    if ("manufacturer" in desc or "industrial" in desc or "factory" in desc or "mill" in desc or
        "lumber" in desc or "cold storage" in desc or "utility" in desc or "utilities" in desc or
        "electric" in desc or "pipeline" in desc or "substation" in desc or "terminal" in desc or
        "gas measuring" in desc or "electric transmission" in desc or "electric distribution" in desc or
        "electric and gas" in desc or "sewage" in desc or "telecommunications" in desc or
        "media studio" in desc or "water supply" in desc or "solid waste" in desc or "junk" in desc or "warehouse" in desc or
        "petroleum" in desc or "cell tower" in desc):
        return "Industrial / Utility"
    # --- Institutional (schools, colleges, libraries, municipal, health, benevolent, cultural, etc) ---
    if ("school" in desc or "college" in desc or "university" in desc or "library" in desc or
        "government" in desc or "police" in desc or "fire" in desc or "municipal" in desc or
        "hospital" in desc or "clinic" in desc or "health" in desc or "auditorium" in desc or
        "culture" in desc or "professional building" in desc or "special schools" in desc or
        "miscellaneous franchise" in desc or "diner" in desc or "spa" in desc or "correctional" in desc):
        return "Institutional / Civic"
    # --- Religious/Charitable/Benevolent ---
    if ("religious" in desc or "benevolent" in desc or "ymca" in desc or "ywca" in desc or
        "home for the aged" in desc or "social organization" in desc or "animal welfare" in desc or
        "welfare" in desc or "cemetery" in desc or "church" in desc or "funeral" in desc):
        return "Religious / Charitable / Social"
    # --- Recreation/Park/Leisure/Cultural ---
    if ("park" in desc or "playground" in desc or "marina" in desc or "athletic" in desc or
        "field" in desc or "stadium" in desc or "recreational" in desc or "entertainment" in desc or
        "public park" in desc or "state park" in desc or "indoor sport" in desc or "indoor rink" in desc or
        "bowling" in desc or "picnic" in desc or "water district" in desc):
        return "Recreation / Park"
    # --- Transportation/Storage/Terminal ---
    if ("railroad" in desc or "transport" in desc or "terminal" in desc or "garage" in desc or
        "truck" in desc or "street" in desc or "highway" in desc or desc == "lot" or "road" in desc):
        return "Transportation / Storage"
    # --- Catch unhandled common residential codes ---
    if (
        "residential" in desc
        and not (
            "1 family" in desc or "2 family" in desc or "3 family" in desc or
            "multiple" in desc or "apartment" in desc or "row" in desc or "vacant" in desc
        )
    ):
        return "Other Residential"
    # --- Limit Other / Miscellaneous category ---
    return "Other / Miscellaneous"

gdf['PROPERTY_CATEGORY'] = gdf['CLASSDSCRP'].apply(aggregate_classdscrp)

In [ ]:
# Ensure proper GeoDataFrame format
gdf = ensure_geodataframe(gdf)

# Create vacant flag based on CLASSDSCRP containing "vacant" (case-insensitive)
gdf['is_vacant'] = gdf['CLASSDSCRP'].str.lower().str.contains("vacant")


# Display property statistics
print(f"🏠 Property Statistics:")
print(f"Total parcels: {len(gdf):,}")
print(f"Vacant parcels: {gdf['is_vacant'].sum():,} ({gdf['is_vacant'].sum()/len(gdf)*100:.1f}%)")

print("\n📋 Property Categories (Standardized):")
category_counts = gdf['PROPERTY_CATEGORY'].value_counts()
for category, count in category_counts.head(10).items():
    pct = count / len(gdf) * 100
    print(f"  {category}: {count:,} ({pct:.1f}%)")

print("\n🏗️ Property Value Statistics:")
print(f"Mean land value: ${gdf['land_value'].mean():,.2f}")
print(f"Mean improvement value: ${gdf['improvement_value'].mean():,.2f}")
print(f"Total land value: ${gdf['land_value'].sum():,.2f}")
print(f"Total improvement value: ${gdf['improvement_value'].sum():,.2f}")


In [ ]:
print("gdf columns:", list(gdf.columns))
gdf_with_exemptions = gdf.copy()
# Exclude parcels where total_exemptions is greater than or equal to CURRENT_TOTAL_VALUE
gdf = gdf[gdf['total_exemptions'] < gdf['CURRENT_TOTAL_VALUE']]
# Split into homestead and non-homestead
homestead_gdf = gdf[gdf['homestead_flag'] == 1].copy()
nonhomestead_gdf = gdf[gdf['homestead_flag'] != 1].copy()

# Set millage rate and recalculate fields
homestead_gdf['millage_rate'] = 7.05
nonhomestead_gdf['millage_rate'] = 15.72

for sub_gdf in [homestead_gdf, nonhomestead_gdf]:
    sub_gdf['exemptions'] = sub_gdf['total_exemptions']
    sub_gdf['improvement_value'] = sub_gdf['CURRENT_TOTAL_VALUE'] - sub_gdf['CURRENT_LAND_VALUE']

from lvt_utils import calculate_current_tax

# Calculate tax for each subset
h_total_revenue, h_second_revenue, homestead_gdf = calculate_current_tax(
    df=homestead_gdf,
    tax_value_col='CURRENT_TOTAL_VALUE',
    millage_rate_col='millage_rate',
    exemption_col='exemptions'
)

nh_total_revenue, nh_second_revenue, nonhomestead_gdf = calculate_current_tax(
    df=nonhomestead_gdf,
    tax_value_col='CURRENT_TOTAL_VALUE',
    millage_rate_col='millage_rate',
    exemption_col='exemptions'
)

# Concatenate back together and preserve row order as much as possible
gdf = pd.concat([homestead_gdf, nonhomestead_gdf]).sort_index()

total_revenue = h_total_revenue + nh_total_revenue
print(f"Calculated total current tax revenue: ${total_revenue:,.2f}")



|                                    | Current Operations    | Cash Capital        | Debt Service     | Total             |
|------------------------------------|----------------------|---------------------|------------------|-------------------|
| **EXPENSE**                        |                      |                     |                  |                   |
| Operating                          | $83,950,699          |                     |                  | $83,950,699       |
| Cash Capital                       |                      | $10,000,000         | $10,000,000      | $10,000,000       |
| Debt Service                       |                      |                     | $81,351,623      | $81,351,623       |
| Tax Reserve                        | $3,955,792           | $471,204            | $932,504         | $5,359,500        |
| **Total**                          | $87,906,491          | $10,471,204         | $82,284,127      | $180,661,822      |
|                                    |                      |                     |                  |                   |
| **REVENUE**                        |                      |                     |                  |                   |
| Operating                          | $0                   |                     |                  | $0                |
| Cash Capital                       |                      | $0                  |                  | $0                |
| Debt Service*                      |                      |                     | $61,561,822      | $61,561,822       |
| **Total**                          | $0                   | $0                  | $61,561,822      | $61,561,822       |
|                                    |                      |                     |                  |                   |
| **TOTAL TAX LEVY**                 | $87,906,491          | $10,471,204         | $20,722,305      | $119,100,000      |
| - Homestead (44.68633%)**          |                      |                     |                  | $53,221,419       |
| - Non-Homestead (55.31367%)**      |                      |                     |                  | $65,878,581       |
|                                    |                      |                     |                  |                   |
| **ASSESSED VALUE**                 |                      |                     |                  | $11,739,074,598   |
| - Homestead                        |                      |                     |                  | $7,548,294,716    |
| - Non-Homestead                    |                      |                     |                  | $4,190,779,882    |
|                                    |                      |                     |                  |                   |
| **TAX RATE**                       |                      |                     |                  |                   |
| - Homestead                        | $5.20                | $0.62               | $1.23            | $7.05             |
| - Non-Homestead                    | $11.60               | $1.38               | $2.74            | $15.72            |

\*Revenues and debt exclusions are recorded here. Revenues related to City School District.
\**Percent of total assessed value.


In [ ]:
homestead_total_taxable = homestead_gdf['CURRENT_TAXABLE_VALUE'].sum()
nonhomestead_total_taxable = nonhomestead_gdf['CURRENT_TAXABLE_VALUE'].sum()
homestead_total_exemptions = homestead_gdf['total_exemptions'].sum()
nonhomestead_total_exemptions = nonhomestead_gdf['total_exemptions'].sum()
print(f"Total taxable value (homestead): ${homestead_total_taxable:,.2f}")
print(f"Total taxable value (non-homestead): ${nonhomestead_total_taxable:,.2f}")
print(f"Total exemptions (homestead): ${homestead_total_exemptions:,.2f}")
print(f"Total exemptions (non-homestead): ${nonhomestead_total_exemptions:,.2f}")


In [ ]:
print("gdf columns:", list(gdf.columns))
# Assuming 'current_tax' and 'TrueTax' columns exist in gdf

print("\n🔎 Comparison between current_tax and TrueTax in gdf:")

# Only keep rows where both values are not null
compare = gdf[['current_tax', 'TrueTax']].dropna()
n = len(compare)

if n == 0:
    print("No properties have both current_tax and TrueTax values.")
else:
    # Calculate masks
    exact_equal = (compare['current_tax'] == compare['TrueTax'])
    pct_diff = (compare['current_tax'] - compare['TrueTax']).abs() / compare['TrueTax'].replace(0, float('nan')).abs()

    percent_exact = (exact_equal.sum() / n) * 100
    percent_within_1 = ((pct_diff <= 0.01).sum() / n) * 100
    percent_within_5 = ((pct_diff <= 0.05).sum() / n) * 100
    percent_within_10 = ((pct_diff <= 0.10).sum() / n) * 100

    print(f"Total compared: {n:,}")
    print(f"Percentage exactly equal: {percent_exact:.2f}%")
    print(f"Percentage within 1%: {percent_within_1:.2f}%")
    print(f"Percentage within 5%: {percent_within_5:.2f}%")
    print(f"Percentage within 10%: {percent_within_10:.2f}%")


In [ ]:
# Prepare data for LVTShift modeling (if not already done during data scraping)
print("🔧 Preparing data for 4:1 split-rate tax modeling...")

print("📊 Running 4:1 split-rate tax calculations for homestead and non-homestead properties separately...")

# Split gdf into homestead and nonhomestead (if not done already above)
# (Assumes `homestead_gdf` and `nonhomestead_gdf` defined earlier code)

# Run split-rate model for homestead
h_land_rate, h_building_rate, h_actual_revenue, h_results_df = model_split_rate_tax(
    df=homestead_gdf,
    land_value_col='CURRENT_LAND_VALUE',
    improvement_value_col='improvement_value', 
    current_revenue=homestead_gdf['current_tax'].sum(),
    land_improvement_ratio=10,  # 4:1 ratio
    exemption_col='exemptions'
)

# Run split-rate model for non-homestead
nh_land_rate, nh_building_rate, nh_actual_revenue, nh_results_df = model_split_rate_tax(
    df=nonhomestead_gdf,
    land_value_col='CURRENT_LAND_VALUE',
    improvement_value_col='improvement_value', 
    current_revenue=nonhomestead_gdf['current_tax'].sum(),
    land_improvement_ratio=10,  # 4:1 ratio
    exemption_col='exemptions'
)

# Copy results back to the respective DataFrames
homestead_gdf['new_tax'] = h_results_df['new_tax']
homestead_gdf['tax_change'] = h_results_df['tax_change']
homestead_gdf['tax_change_pct'] = h_results_df['tax_change_pct']

nonhomestead_gdf['new_tax'] = nh_results_df['new_tax']
nonhomestead_gdf['tax_change'] = nh_results_df['tax_change']
nonhomestead_gdf['tax_change_pct'] = nh_results_df['tax_change_pct']

# Concatenate back together to restore main DataFrame
gdf = pd.concat([homestead_gdf, nonhomestead_gdf]).sort_index()

# Show summary for both
total_actual_revenue = h_actual_revenue + nh_actual_revenue
total_current_revenue = homestead_gdf['current_tax'].sum() + nonhomestead_gdf['current_tax'].sum()

print(f"✅ Split-rate calculations completed using LVTShift tools (modeled separately by homestead flag)")
print(f"   Homestead - Land rate: ${h_land_rate*1000:.3f} per $1,000, Building rate: ${h_building_rate*1000:.3f} per $1,000")
print(f"   Non-homestead - Land rate: ${nh_land_rate*1000:.3f} per $1,000, Building rate: ${nh_building_rate*1000:.3f} per $1,000")
print(f"   Combined modeled revenue: ${total_actual_revenue:,.2f} | Original current: ${total_current_revenue:,.2f}")
print(f"   Revenue accuracy: {total_actual_revenue/total_current_revenue*100:.4f}%")



In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)
display(gdf.head())

In [ ]:
# Calculate comprehensive tax impact summary by property category
category_summary = calculate_category_tax_summary(
    df=gdf,
    category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    pct_threshold=10.0
)

# Print formatted summary
print_category_tax_summary(
    summary_df=category_summary,
    title="4:1 Split-Rate Tax Impact by Property Category - Syracuse, NY",
    pct_threshold=10.0
)

# Display the detailed summary table
display(category_summary)


In [ ]:
# Property category impact chart (Spokane style, sorted, ignore 0% median)
import matplotlib.pyplot as plt
import numpy as np

# Filter out categories where median tax change percent is exactly 0
filtered = category_summary[category_summary['median_tax_change_pct'] != 0].copy()

# Only include categories with property_count > 0 (optional, but safe)
filtered = filtered[filtered['property_count'] > 0]

# Sort by median_pct_change ascending (like Spokane)
categories = filtered['PROPERTY_CATEGORY'].tolist()
counts = filtered['property_count'].tolist()
median_pct_change = filtered['median_tax_change_pct'].tolist()
median_dollar_change = filtered['median_tax_change'].tolist()
total_tax_change = (
    filtered['total_tax_change_dollars'].tolist()
    if 'total_tax_change_dollars' in filtered.columns
    else (filtered['mean_tax_change'] * filtered['property_count']).tolist()
)

# Sort by median_pct_change ascending
sorted_idx = np.argsort(median_pct_change)
categories = [categories[i] for i in sorted_idx]
counts = [counts[i] for i in sorted_idx]
median_pct_change = [median_pct_change[i] for i in sorted_idx]
median_dollar_change = [median_dollar_change[i] for i in sorted_idx]
total_tax_change = [total_tax_change[i] for i in sorted_idx]

# Custom color: anything above 0 is dark red, below 0 is green
bar_colors = []
for val in median_pct_change:
    if val > 0:
        bar_colors.append("#8B0000")  # dark red
    else:
        bar_colors.append("#228B22")  # professional green

# Bar settings
bar_height = 0.75
fig_height = len(categories) * 0.8 + 1.2
right_col_pad = 120  # more padding for right column
fig, ax = plt.subplots(figsize=(17, fig_height))  # wider for right column

y = np.arange(len(categories))

# Draw bars
ax.barh(
    y, median_pct_change, color=bar_colors, edgecolor='none',
    height=bar_height, alpha=0.92, linewidth=0, zorder=2
)

# Remove all spines and ticks for a clean look
for spine in ax.spines.values():
    spine.set_visible(False)
ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

# Adjusted vertical spacing
cat_offset = 0.18   # less space between category and median
med_offset = -0.03  # median just below category
count_offset = -0.23  # more space below median for parcels

# For right column: position for total tax change
max_abs = max(abs(min(median_pct_change)), abs(max(median_pct_change)))
right_col_x = max_abs + right_col_pad

# Add Net Change header at the top of the right column
ax.text(
    right_col_x, len(categories) - 0.5, "Net Change", va='bottom', ha='left',
    fontsize=15, fontweight='bold', color='black', fontname='Arial'
)

for i, (cat, val, count, med_dol, tot_change) in enumerate(zip(categories, median_pct_change, counts, median_dollar_change, total_tax_change)):
    # Format median dollar and percent change together
    if med_dol >= 0:
        med_dol_str = f"${med_dol:,.0f}"
    else:
        med_dol_str = f"-${abs(med_dol):,.0f}"
    pct_str = f"{val:+.1f}%"
    median_combo = f"Median: {med_dol_str}, {pct_str}"

    # Position: right of bar for positive, left for negative
    if val < 0:
        xpos = val - 2.5
        ha = 'right'
    else:
        xpos = val + 2.5
        ha = 'left'
    # Category name (bold, bigger)
    ax.text(
        xpos, y[i]+cat_offset, cat, va='center', ha=ha,
        fontsize=14, fontweight='bold', color='#222',
        fontname='Arial'
    )
    # Median (dollar + percent, bold, black, just below category)
    ax.text(
        xpos, y[i]+med_offset, median_combo, va='center', ha=ha,
        fontsize=12, fontweight='bold', color='black',
        fontname='Arial'
    )
    # Count (bold, smaller, below median)
    ax.text(
        xpos, y[i]+count_offset, f"{count:,} parcels", va='center', ha=ha,
        fontsize=11, fontweight='bold', color='#888',
        fontname='Arial'
    )
    # Net change column, always right-aligned in a new column, black text, no "Total:"
    if tot_change >= 0:
        tot_change_str = f"${tot_change:,.0f}"
    else:
        tot_change_str = f"-${abs(tot_change):,.0f}"
    ax.text(
        right_col_x, y[i], tot_change_str, va='center', ha='left',
        fontsize=13, fontweight='bold', color='black',
        fontname='Arial'
    )

# Set x limits for symmetry, make bars longer, and leave space for right column
ax.set_xlim(-right_col_x, right_col_x + 60)

# Remove axis labels/ticks
ax.set_yticks([])
ax.set_xticks([])

plt.tight_layout()
plt.show()


In [ ]:
# Output the count of CLASSDSCRP for PROPERTY_CATEGORY 'Other / Miscellaneous'
other_misc_mask = gdf['PROPERTY_CATEGORY'] == 'Other / Miscellaneous'
other_misc_class_counts = gdf.loc[other_misc_mask, 'CLASSDSCRP'].value_counts()
print("CLASSDSCRP counts for PROPERTY_CATEGORY = 'Other / Miscellaneous':")
print(other_misc_class_counts)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
output_summary = category_summary
# Summarize output by PROPERTY_CATEGORY, using the output_summary DataFrame (as in Spokane)
summary_filtered = output_summary[output_summary['property_count'] > 50].copy()
summary_sorted = summary_filtered.sort_values('pct_increase_gt_threshold', ascending=True)

categories_sorted = summary_sorted['PROPERTY_CATEGORY'].tolist()
pct_increase_sorted = summary_sorted['pct_increase_gt_threshold'].tolist()
pct_decrease_sorted = summary_sorted['pct_decrease_gt_threshold'].tolist()

# Convert to integers for display
pct_increase_int_sorted = [int(round(x)) for x in pct_increase_sorted]
pct_decrease_int_sorted = [int(round(x)) for x in pct_decrease_sorted]

y = np.arange(len(categories_sorted))

fig, ax = plt.subplots(figsize=(8, 6))

color_increase = "#8B0000"
color_decrease = "#228B22"

# Plot decreases (green, left)
ax.barh(
    y, 
    [-v for v in pct_decrease_sorted], 
    color=color_decrease, 
    edgecolor='none', 
    height=0.7
)
# Plot increases (red, right)
ax.barh(
    y, 
    pct_increase_sorted, 
    color=color_increase, 
    edgecolor='none', 
    height=0.7
)

# Add percent labels
for i, (inc, dec) in enumerate(zip(pct_increase_int_sorted, pct_decrease_int_sorted)):
    if dec > 0:
        ax.text(
            -dec - 2, y[i], f"{dec}%", 
            va='center', ha='right', 
            fontsize=8, fontweight='normal', color=color_decrease, fontname='Arial'
        )
    if inc > 0:
        ax.text(
            inc + 2, y[i], f"{inc}%", 
            va='center', ha='left', 
            fontsize=8, fontweight='normal', color=color_increase, fontname='Arial'
        )

# Add category label, right of increase bar
for i, (cat, inc) in enumerate(zip(categories_sorted, pct_increase_sorted)):
    xpos = inc + 18 if inc > 0 else 18
    ax.text(
        xpos, y[i], cat, 
        va='center', ha='left', 
        fontsize=9, fontweight='bold', color='#222', fontname='Arial'
    )

# Minimalist: remove all spines, ticks, axis lines
for spine in ax.spines.values():
    spine.set_visible(False)
ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
ax.set_yticks([])
ax.set_xticks([])
ax.set_ylabel('')
ax.set_xlabel('')
ax.set_title('')

# Symmetrize
max_val = max(max(pct_increase_sorted), max(pct_decrease_sorted))
ax.set_xlim(-max_val-20, max_val+48)

# Custom titles
title_fontsize = 10
title_color = 'black'
title_fontweight = 'normal'
title_fontname = 'Arial'
title_y = len(categories_sorted) - 0.2

left_title_x = -max_val * 0.45
ax.text(
    left_title_x, title_y, 
    "Percent of parcels\ndecreasing >10%", 
    ha='center', va='bottom', fontsize=title_fontsize, fontweight=title_fontweight, 
    color=title_color, fontname=title_fontname, 
    bbox=dict(facecolor='white', edgecolor='none', boxstyle='round,pad=0.15')
)
right_title_x = max_val * 0.45
ax.text(
    right_title_x, title_y, 
    "Percent of parcels\nincreasing >10%", 
    ha='center', va='bottom', fontsize=title_fontsize, fontweight=title_fontweight, 
    color=title_color, fontname=title_fontname, 
    bbox=dict(facecolor='white', edgecolor='none', boxstyle='round,pad=0.15')
)

plt.tight_layout()
plt.show()


# census

In [ ]:
# Get census data for Monroe County, NY (Rochester) - FIPS code: 36055
print("📊 Loading Census data for Monroe County, NY...")
df  = gdf
try:
    census_data, census_boundaries = get_census_data_with_boundaries(
        fips_code="36055",  # Monroe County, NY
        year=2022
    )
    print("GOT CENSUS DATA")
    # Set CRS for census boundaries before merging
    census_boundaries = census_boundaries.set_crs(epsg=4326)  # Assuming WGS84 coordinate system

    # Ensure our parcel data is in the same CRS
    if df.crs != census_boundaries.crs:
        df = df.to_crs(census_boundaries.crs)

    # Merge census data with our parcel boundaries
    df = match_to_census_blockgroups(
        gdf=df,
        census_gdf=census_boundaries
    )

    print(f"✅ Census data integration complete!")
    print(f"Number of census block groups: {len(census_boundaries)}")
    print(f"Number of census data records: {len(census_data)}")
    print(f"Number of parcels with census data: {len(df)}")

    # Display new columns added
    census_cols = [col for col in df.columns if col in ['median_income', 'minority_pct', 'black_pct', 'total_pop', 'census_block_group']]
    print(f"Census columns added: {census_cols}")

except Exception as e:
    print(f"❌ Error loading census data: {e}")


In [ ]:
# Analyze tax impacts by income quintiles (similar to Spokane analysis), residential property only
print("📊 Analyzing tax impacts by neighborhood income quintiles (Residential parcels only)...")

if 'median_income' in df.columns:
    # Filter to only rows where property_category contains 'residential' (case-insensitive)
    df_residential = df[
        df['PROPERTY_CATEGORY'].str.contains('residential', case=False, na=False)
    ].copy()
    
    # Filter out parcels with missing or non-positive income data
    df_with_income = df_residential[(df_residential['median_income'].notna()) & (df_residential['median_income'] > 0)].copy()
    
    # Create income quintiles
    df_with_income['income_quintile'] = pd.qcut(
        df_with_income['median_income'], 
        5, 
        labels=["Q1 (Lowest)", "Q2", "Q3", "Q4", "Q5 (Highest)"]
    )
    
    # Calculate summary statistics by quintile
    quintile_summary = df_with_income.groupby('income_quintile').agg({
        'tax_change': ['count', 'mean', 'median'],
        'tax_change_pct': ['mean', 'median'],
        'median_income': 'mean',
        'current_tax': 'mean'
    }).round(2)
    
    # Flatten column names
    quintile_summary.columns = ['_'.join(col).strip() for col in quintile_summary.columns]
    quintile_summary = quintile_summary.reset_index()
    
    print(f"✅ Income quintile analysis complete (Residential only)")
    print(f"Residential parcels with income data: {len(df_with_income):,} ({len(df_with_income)/len(df_residential)*100:.1f}% of residential)")
    
    display(quintile_summary)
    
    # Create visualization
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # Plot 1: Median tax change percentage by income quintile
    quintile_data = quintile_summary.copy()
    quintile_data['median_tax_change_pct'] = quintile_data['tax_change_pct_median']
    
    bars1 = ax1.bar(
        quintile_data['income_quintile'],
        quintile_data['median_tax_change_pct'],
        color='steelblue',
        alpha=0.7
    )
    
    ax1.set_title('Median Tax Change % by Income Quintile (Residential)', fontweight='bold', pad=20)
    ax1.set_ylabel('Median Tax Change (%)')
    ax1.set_xlabel('Income Quintile')
    ax1.grid(True, alpha=0.3)
    
    # Add value labels on bars
    for bar, val in zip(bars1, quintile_data['median_tax_change_pct']):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{val:.1f}%', ha='center', va='bottom', fontweight='bold')
    
    # Plot 2: Average neighborhood income by quintile
    bars2 = ax2.bar(
        quintile_data['income_quintile'],
        quintile_data['median_income_mean'],
        color='green',
        alpha=0.7
    )
    
    ax2.set_title('Average Neighborhood Income by Quintile (Residential)', fontweight='bold', pad=20)
    ax2.set_ylabel('Average Median Income ($)')
    ax2.set_xlabel('Income Quintile')
    ax2.grid(True, alpha=0.3)
    
    # Format y-axis as currency
    ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))
    
    # Add value labels on bars
    for bar, val in zip(bars2, quintile_data['median_income_mean']):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1000,
                f'${val:,.0f}', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
else:
    print("❌ Census income data not available - skipping quintile analysis")


In [ ]:
# Create a BAR CHART using the output from the above calculated quintile_summary DataFrame
# (from @file_context_0, which is referred to as `quintile_summary` here)

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Set style
sns.set_theme(style="whitegrid", font_scale=1.10)

# Use the correct, already calculated medians for each quintile (from previous code)
vals = quintile_summary['tax_change_pct_median']
labels = quintile_summary['income_quintile']

fig, ax = plt.subplots(figsize=(10, 6))

# Color palette (blue to green, left to right)
colors = sns.color_palette("viridis", n_colors=len(vals))

bars = ax.bar(
    labels,
    vals,
    color=colors,
    edgecolor='black',
    width=0.8,
    alpha=0.85
)

ax.set_title('Median Tax Change % by Income Quintile (Residential)', fontweight='bold', pad=20)
ax.set_ylabel('Median Tax Change (%)')
ax.set_xlabel('Income Quintile')
ax.grid(axis='y', alpha=0.3, linestyle='--')

# Add value labels on bars, centered near top of each bar
for bar, val in zip(bars, vals):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + np.sign(val) * 0.5,
        f'{val:.1f}%',
        ha='center', va='bottom' if val >= 0 else 'top',
        fontweight='bold'
    )

plt.tight_layout()
plt.show()


In [ ]:

# --- Export rochester.parquet with same columns as Syracuse ---
import os
import numpy as np

# Ensure df is a GeoDataFrame with correct CRS and columns
export_gdf = df.copy()

# Create full_exmp flag if taxable_value is < 1
export_gdf['full_exmp'] = export_gdf['taxable_value'] < 1


# 1. Exemption flag (1 if property is fully exempt, 0 otherwise)
# The Rochester data field is 'full_exmp' (boolean flag as used previously)
export_gdf['exemption_flag'] = (export_gdf['full_exmp'] == True).astype(int)

# 2. Property category
# Use existing field for main property category
export_gdf['property_land_use_category'] = export_gdf['PROPERTY_CATEGORY']

# 3. Refined property/land use category (Vacant, Parking Lot, Underdeveloped)
def categorize_property_refined_rochester(row):
    cat = str(row['PROPERTY_CATEGORY']).lower()
    if "vacant" in cat:
        return "Vacant"
    elif "parking" in cat:
        return "Parking Lot"
    # Underdeveloped: improvement < half (land + improvement) AND non-vacant/parking
    try:
        land_val = float(row['land_value']) if not pd.isnull(row['land_value']) else 0
        imp_val = float(row['improvement_value']) if not pd.isnull(row['improvement_value']) else 0
        # Only flag as underdeveloped if it's not already in above categories
        if imp_val < 0.5 * (land_val + imp_val) and imp_val > 0 and land_val > 100:  # avoid flagging if both zero
            return "Underdeveloped"
    except Exception:
        pass
    return None

export_gdf['property_land_use_refined'] = export_gdf.apply(categorize_property_refined_rochester, axis=1)

# 4. Area (in square feet) from geometry
# Prefer shape area if available and units are square feet, otherwise compute from geometry
if 'Shape__Area' in export_gdf.columns:
    # Double-check units, but often ESRI Shape__Area is in sq feet for New York exports, else fallback
    # We'll check against geometry (since geometry is projected to WGS84, area may be off) so use Shape__Area if plausible
    if export_gdf['Shape__Area'].max() > 100000:  # crude check: if numbers look like sq feet not sq meters
        export_gdf['area_sqft'] = export_gdf['Shape__Area']
    else:
        export_gdf['area_sqft'] = export_gdf['Shape__Area'] * 10.7639  # assume they're meters, convert to sq ft
else:
    # Compute from geometry (will be approximate with WGS84, but best fallback)
    # Sometimes the geometry is still in a projected system; if not, force to EPSG:2263 (NYS Plane) for area.
    if hasattr(export_gdf, "crs") and export_gdf.crs is not None and export_gdf.crs.to_epsg() != 2263:
        try:
            export_gdf = export_gdf.to_crs(2263)
        except Exception:
            pass  # fallback: stay in original CRS
    export_gdf['area_sqft'] = export_gdf.geometry.area

# 5. Current tax per square foot
export_gdf['current_tax_per_sqft'] = np.where(
    export_gdf['area_sqft'] > 0,
    export_gdf['current_tax'] / export_gdf['area_sqft'],
    0
)

# 6. Land value per square foot
export_gdf['land_value_per_sqft'] = np.where(
    export_gdf['area_sqft'] > 0,
    export_gdf['land_value'] / export_gdf['area_sqft'],
    0
)

# 7. Improvement value per square foot
export_gdf['improvement_value_per_sqft'] = np.where(
    export_gdf['area_sqft'] > 0,
    export_gdf['improvement_value'] / export_gdf['area_sqft'],
    0
)

# 8. Rename columns for export as required by Syracuse format
columns_to_export = [
    'geometry',
    'exemption_flag',
    'property_land_use_category',
    'property_land_use_refined',
    'current_tax',
    'current_tax_per_sqft',
    'land_value',
    'land_value_per_sqft',
    'improvement_value',
    'improvement_value_per_sqft'
]

rename_dict = {
    'land_value': 'current_full_land_value'
}

export_final = export_gdf[columns_to_export].rename(columns=rename_dict)

# Ensure geometry validity before reprojecting to WGS84/EPSG:4326
export_final['geometry'] = export_final['geometry'].apply(
    lambda g: g if g is None or g.is_valid else g.buffer(0)
)

# Force output CRS to WGS84 (EPSG:4326)
if not hasattr(export_final, "crs") or export_final.crs is None or export_final.crs.to_epsg() != 4326:
    try:
        export_final = export_final.set_geometry("geometry")
    except Exception:
        pass
    try:
        export_final = export_final.to_crs(epsg=4326)
        print("Converted to EPSG:4326")
    except Exception as e:
        print("Warning: Could not reproject to EPSG:4326:", str(e))

# Save as Parquet to Downloads
output_filename = os.path.expanduser("~/Downloads/rochester.parquet")
export_final.to_parquet(output_filename, index=False)
print(f"\n✅ Saved rochester.parquet to Downloads")
print("Saved columns:", export_final.columns.tolist())
print("Property refined category counts:")
print(export_final['property_land_use_refined'].value_counts(dropna=False))
print("Property category counts:")
print(export_final['property_land_use_category'].value_counts().head(10))

print(f"\n👀 First 5 rows of exported data:")
display(export_final.head())


In [ ]:


print(f"\\n✅ Analysis complete! Review the visualizations above for detailed insights.")


In [ ]:
# Export standardized CSV — rochester
import sys
sys.path.insert(0, '../..')
from lvt_utils import save_standard_export

# Rochester models homestead (7.05 mill) and nonhomestead (15.72 mill) separately at 10:1 ratio
# Use revenue-weighted average land/improvement millage
_h_rev = homestead_gdf['current_tax'].sum()
_nh_rev = nonhomestead_gdf['current_tax'].sum()
_total_rev = _h_rev + _nh_rev
_land_millage = (h_land_rate * _h_rev + nh_land_rate * _nh_rev) / _total_rev
_imp_millage = (h_building_rate * _h_rev + nh_building_rate * _nh_rev) / _total_rev

gdf['taxable_land_value'] = gdf['CURRENT_LAND_VALUE'].clip(lower=0)
gdf['taxable_improvement_value'] = gdf['improvement_value'].clip(lower=0)

save_standard_export(
    df=gdf,
    city='rochester',
    output_path='../../analysis/data/rochester.csv',
    model_type='split_rate:10.0',
    land_millage=float(_land_millage),
    improvement_millage=float(_imp_millage),
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
)